# Multi-view-clustering BDPC on Reacher

Equation-based implementation of **Shuangyu Han, Yitao Yan, Jie Bao and Biao Huang, Big Data-driven Predictive Control Using Multi-view Clustering, ACC 2024, pp. 5100–5105**. This follows the supplied six-page conference paper, not later contraction/dissipativity variants. It is **not authors' source code or an exact reproduction of their process experiment**.

**Same offline dataset as Atlas; all eight physical outputs and a fixed p=8 reference; one-step prediction; no graph guidance.** All 200 original 200-sample offline episodes are recovered and all original Hankel overlaps checked. No new training trajectories are generated.

**Run All is safe by default:** preparation, smoke testing and the 100-seed panel are all disabled. **The historical XY-only tuning did not find a working controller. The new p=8 development setup is untuned; inspect its model audit before a full panel.** No existing comparison results are overwritten. **Development protocol: seeds 211–310, 300 steps each. MVC retains C=5 and SVD cutoff 0.01; K=64/rank=53 identifies only the source of the reference goals.**

Source PDF: `/Users/kumiko/Downloads/Big_Data-driven_Predictive_Control_Using_Multi-view_Clustering.pdf`; SHA256 `53e4347222752e981d8fac360ebdfc4f6741739eb1670fe7de4b1f809d508975`.


## What is retained from the paper

| Component | Implementation / paper location |
|---|---|
| Global scaling | Training-only channel z-scores, as in Section IV-B; unique samples, not repeated overlapping windows |
| View 1 | Pairwise Euclidean distances between complete equal-length input/output trajectories, Eq. (6) |
| View 2 | Principal-angle distance with unequal-rank penalty, Eq. (8), not chordal distance |
| Clustering | Joint nonnegative matrix factorization, multiplicative U/V updates and weighted consensus, Algorithm 1 |
| Cluster representative | Recorded trajectory with maximum consensus loading for each cluster, Section III-A; not a mean or medoid |
| Online selection | Weighted, separately sum-normalized two-view past-trajectory matching error, Eq. (9); no reference in selection |
| Local predictor | Hankel of the selected representative trajectory, Eqs. (10)–(11); not pooled Atlas charts |
| Coordinates | Deviations from the representative's final L-sample segment, restored by adding its final sample, Eq. (11d) |
| Horizon | L=7, L_p=6, L_f=1, matching the paper's example |
| Clusters / consensus weights | C=5, lambda_1=lambda_2=0.1, matching the paper's example |
| Optimization | Weighted output/reference and input/reference norms plus L2 past-output slack; past inputs exact |
| Not added | No sum(g)=1, coefficient penalty/bounds, projection regularizer, input smoothing, graph, IK, rescue or refinement loop |

**Here Delta means deviation from a cluster representative, not time differences.** This is not contextual DPC's input-increment augmentation.

Unlike the previous raw-trajectory baselines, this paper explicitly discusses low-rank SVD approximation. We zero small singular values offline but keep the representative Hankel's **194 columns** and **194 g coefficients**. There is no Atlas-style reduction to rank-53 coordinates. With the 1% cutoff, retained ranks vary by trajectory rather than being fixed.


## Necessary Reacher adaptations and unspecified details

These choices are explicit so the implementation is not mistaken for an author-verified reproduction.

1. **Outputs/reference:** p=8, ordered [cos(q1), cos(q2), sin(q1), sin(q2), dq1, dq2, fingertip x, fingertip y]. All eight channels enter the offline models, past histories, output slack and tracking objective. Fixed goals are reconstructed from the retained K=64/rank-53 development summary: its saved offline column supplies the posture, velocities are zero and XY is the seeded environment target. No test-seed results, Atlas controller, graph, or new goal optimization are used. Reacher-v5, 12 zero-input warmup steps, seeds 211–310, **300 controls**, one numerical thread; physical torque reference u_r=0.
2. **Low-rank cutoff:** the paper says to zero small singular values but supplies no rule. Default relative cutoff is **0.01 times the largest singular value**, chosen from offline spectra only, not test rollouts. Those earlier spectrum checks were for p=2 (0.01 gave ranks 21–24), not p=8. The new p=8 spectra and ranks are computed during offline preparation; rank 53 is NOT imposed on MVC. The cutoff is configurable, changes the algorithm's local models and must be disclosed.
3. **Online angle embedding:** Eq. (9) does not specify Hankel depth for short past segments. We compare H_(L_p) of the L_p measured samples with H_(L_p) of each representative's matching past segment. These are one-column spaces. No future observation or target is used. This is a declared interpretation, not a verified author implementation.
4. **Center-relative model:** following the literal Eqs. (10)–(11), H_L(w_c) is constructed from the globally standardized representative, while the online right-hand side is center-relative. We do not silently replace it with a Hankel of centered differences or a pooled cluster model. The paper does not give additional model-construction code.
5. **Norm convention:** Eq. (11a) prints weighted 2-norms and an unsquared slack 2-norm. We interpret the weighted norm as sqrt(e^T Lambda e) and do not silently square these terms. This is a conic problem solved with Clarabel, not a pure QP. The paper names YALMIP but does not specify a numerical solver or define its weighted-norm notation further.
6. **Starting weights, not tuned Reacher parameters:** Lambda_y=400 I_8 in standardized output coordinates, Lambda_u=I_2, lambda_epsilon=1400. This extends the old scalar output weight to all eight channels; q_joint, q_dq and q_xy are separately editable. These are development starting values, not contextual-DPC weights or validated Reacher parameters. No claim of good Reacher tuning is made.
7. **NMF numerical conventions:** entrywise L1 view normalization; seeded positive initialization; relative objective tolerance and finite inner/outer iteration limits. Representatives follow the printed argmax rule even if duplicates or empty assigned clusters occur. A zero distance view or failed convergence raises an error instead of silently switching algorithms.
8. **Actuator limits:** Eq. (11) prints no input bounds, so none are inserted into the optimization. Reacher's physical ±1 saturation is applied outside the optimizer, with both commanded and applied torques recorded. Predictions correspond to the commanded torque; saturation can therefore invalidate them. A rejected solve aborts that rollout and counts as a failure; no extra fallback controller is substituted.

**Do not call this exactly the authors' algorithm on Reacher without these qualifications.** An exact implementation would require clarification from the authors on the unspecified conventions, especially the SVD cutoff, online angle embedding and weighted norm.


## Historical XY-only implementation check (not a p=8 result)

The previous p=2 implementation was checked on the full offline dataset and an **8-step seed-211 smoke test**, saved separately in `experiments/reacher/mvc_bdpc_implementation_check_final_v2`. Joint NMF converged after 886 outer iterations. The printed representative rule produced 4 distinct representative episodes for 5 cluster columns (two columns selected the same episode); two hard-assignment clusters were empty. These outcomes were retained, not repaired with a different clustering algorithm.

All 8 optimization steps were accepted, but **6/8 commands saturated** and target error rose from 57.24 to 302.15 mm. This is a warning about the untuned one-step Reacher adaptation, **not evidence of successful control**. Inspect the modeling conventions and weights before spending time on a full panel. This does not measure the new p=8 version. No p=8 development panel is launched by this update.

Automated tests cover Hankel indexing, original coefficient count after SVD, principal-angle/rank distance, NMF normalization and consensus, representative selection, past-only matching, center offsets, the unsquared-norm objective, absence of extra constraints, and rejected-solve behavior.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import os, sys, json, signal, subprocess, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/atlas_deepc/benchmarks/mvc_bdpc_terminal8.py').is_file())
sys.path.insert(0, str(ROOT / 'src'))
from atlas_deepc.benchmarks import mvc_bdpc_terminal8 as mvc
importlib.reload(mvc)
DATA = ROOT / 'data/reacher/select_iid_offline_hankel.npy'
RESULT_DIR = mvc.OUTPUT
SMOKE_DIR = ROOT / 'experiments/reacher/mvc_bdpc_p8_smoke_v1'
SEED_START, SEED_STOP, STEPS = 211, 310, 300
ACKNOWLEDGE_IMPLEMENTATION_CONVENTIONS = False
PREPARE_OFFLINE = False
RUN_SMOKE = False
RUN_EXPERIMENT = False
config = mvc.Config(past=6, clusters=5, lambda_view1=0.1, lambda_view2=0.1,
                    q_joint=400.0, q_dq=400.0, q_xy=400.0, r=1.0, lambda_epsilon=1400.0, svd_rtol=0.01)
config.validate()
display(pd.DataFrame([asdict(config)]))
print('MVC: C=5, relative SVD cutoff=0.01 (not K=64/rank=53).')
print('Panel:', SEED_START, 'through', SEED_STOP, '| T_sim =', STEPS, '| p = 8')
print('Data:', DATA)
print('Results:', RESULT_DIR)
print('No execution until the relevant switches and acknowledgement are enabled.')


In [ ]:
def launch(directory, seed_start=None, seed_stop=None, steps=None, prepare_only=False):
    seed_start = SEED_START if seed_start is None else seed_start
    seed_stop = SEED_STOP if seed_stop is None else seed_stop
    steps = STEPS if steps is None else steps
    if not ACKNOWLEDGE_IMPLEMENTATION_CONVENTIONS:
        raise RuntimeError('Read Necessary Reacher adaptations, then acknowledge them in Settings.')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    config_file = directory / 'requested_config.json'
    if config_file.exists() and json.loads(config_file.read_text()) != asdict(config):
        raise RuntimeError('Settings changed: use a new output directory to preserve previous results.')
    mvc.utils.write_json(config_file, asdict(config))
    command = [sys.executable, '-u', '-m', 'atlas_deepc.benchmarks.mvc_bdpc_terminal8',
               '--config', str(config_file), '--data', str(DATA), '--output', str(directory),
               '--seed-start', str(seed_start), '--seed-stop', str(seed_stop), '--steps', str(steps)]
    if prepare_only: command.append('--prepare-only')
    run_env = os.environ.copy()
    run_env['PYTHONPATH'] = str(ROOT / 'src')
    for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
        run_env[name] = '1'
    print('Launching:', ' '.join(command))
    process = subprocess.Popen(command, cwd=ROOT, env=run_env, stdin=subprocess.DEVNULL, start_new_session=True)
    try:
        code = process.wait()
        if code: raise subprocess.CalledProcessError(code, command)
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try: process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM)
            process.wait(timeout=10)
        print('Stopped. Completed seed files remain reusable; the interrupted seed restarts.')
        raise


## Historical XY-only failure diagnosis

These are p=2 results, retained as historical evidence; they are not results of the new p=8 setup. Adding outputs alone is not a guarantee of successful control.

On October 15, 2026, 16 weight settings (including the original) were tested on seed 211 for up to 200 steps. The original and R=4 settings were also tested on seeds 222 and 258 for 320 steps. None of the 20 attempted rollouts met strict success; rejected solves count as unsuccessful. This is a small diagnostic study, not a 100-seed success-rate estimate.

Only Q_xy, R and lambda_epsilon changed. The offline model was reused exactly, with its hash checked. Strong input weights often caused virtually zero actions; weak slack weights allowed predictions to reach the target without corresponding physical movement; other settings saturated or encountered numerical failure. **Defaults were not relabeled as tuned and no failed candidate was promoted.**

A structural check found rank([H_p; U_f]) = rank(H_p) in all five representative models, where H_p contains all past input/output rows. The future-input component outside the past row space is below 2e-12 in relative norm. Thus, fixing all past measurements fixes the future input; the optimizer must use output slack to change its input. Weight changes cannot restore independent future-input directions. This concerns the **current adaptation and its SVD/model-construction conventions**, not a claim that the authors' method cannot work. Model construction should be audited before expanding the search.

Saved evidence: `experiments/reacher/mvc_bdpc_control_tuning_v1/tuning_report.json`. The cell below only reads the report; it does not tune or run simulations.


In [ ]:
tuning_report_path = ROOT / 'experiments/reacher/mvc_bdpc_control_tuning_v1/tuning_report.json'
if tuning_report_path.exists():
    tuning_report = json.loads(tuning_report_path.read_text())
    print('Working parameter set found:', tuning_report['working_parameter_set_found'])
    display(pd.DataFrame(tuning_report['records']).drop(columns=['source']))
    display(pd.DataFrame(tuning_report['model_geometry']))
else:
    print('No saved tuning report found.')


## 1. Prepare and inspect the offline model

Set `ACKNOWLEDGE_IMPLEMENTATION_CONVENTIONS=True` and `PREPARE_OFFLINE=True` in Settings. This validates/reconstructs all 100 development references, then fits the p=8 clustering/model and saves `offline_model.npz`, `offline_audit.json` and a provenance manifest. It does not simulate any control steps. The reference setup only resets environments to recover the seeded targets; it does not perform control rollouts. A different setting/source/data version requires a new result directory. Inspect `model_geometry` in the audit: a zero rank increase after adding future inputs means the structural warning remains even with p=8.


In [ ]:
if PREPARE_OFFLINE:
    launch(RESULT_DIR, prepare_only=True)
else:
    print('Offline preparation disabled.')


In [ ]:
cache = RESULT_DIR / 'offline_model.npz'
if cache.exists():
    with np.load(cache) as a:
        audit = json.loads(a['audit'].item())
        display(pd.DataFrame([{'cluster': j, 'representative_episode': int(i),
                               'assigned_trajectories': audit['cluster_sizes'][j], 'retained_rank': int(a['ranks'][i])}
                              for j, i in enumerate(a['representatives'])]))
        display(pd.DataFrame(audit['model_geometry']))
        print('Distinct representatives:', audit['distinct_representatives'], '/', config.clusters)
        print('NMF converged:', audit['nmf_converged'], '| offline seconds:', audit['offline_setup_seconds'])
        fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
        for j in (0, 1):
            im = axes[j].imshow(a['distances'][j], aspect='auto')
            axes[j].set_title(['Euclidean view', 'Principal-angle view'][j])
            fig.colorbar(im, ax=axes[j])
        axes[2].semilogy(a['nmf_objective'])
        axes[2].set_title('Joint NMF objective'); axes[2].set_xlabel('Outer iteration')
        axes[3].plot(np.sort(a['ranks']))
        axes[3].set_title('Retained trajectory ranks'); axes[3].set_xlabel('Sorted episode')
        fig.tight_layout(); plt.show()
else:
    print('Prepare the offline model first. No cached model loaded.')


## 2. Short smoke test (optional)

Enable `RUN_SMOKE=True` for seed 211 and 8 controlled steps in a separate directory. This tests execution, not success or controller tuning. Review saturation and predicted-versus-realized motion before a long run.


In [ ]:
if RUN_SMOKE:
    launch(SMOKE_DIR, seed_start=211, seed_stop=211, steps=8)
    display(pd.DataFrame(json.loads((SMOKE_DIR / 'summary.json').read_text())['results']))
else:
    print('Smoke test disabled.')


## 3. Run / resume the 100-seed panel

For **100 development rollouts × 300 steps**, set `ACKNOWLEDGE_IMPLEMENTATION_CONVENTIONS=True` and `RUN_EXPERIMENT=True` after reviewing the conventions and smoke test. Existing completed seed files are reused. Rejected solves abort the affected seed without fallback and remain marked as unsuccessful. Interrupt preserves completed seeds. This starts only this BDPC method, not Atlas or either earlier comparison.

The starting weights are **not tuned**, so this first panel is diagnostic, not a final paper comparison.


In [ ]:
if RUN_EXPERIMENT:
    launch(RESULT_DIR)
else:
    print('Full experiment disabled. No simulation launched.')


## 4. Saved results and modular timing

Success uses the existing comparison rule: BOTH last-30 and last-100 windows have mean fingertip error below 10 mm and maximum below 20 mm. Short smoke tests are not success evaluations. Aborted runs are failures; error/RMSE aggregates below explicitly use full rollouts only.

Per-step files preserve physical y (all eight baseline measurements for later analysis), applied u, unconstrained command u, predicted p=8 outputs (and fingertip x/y), fixed p=8 reference, target, selected cluster/representative, both distance scores, matching error, all g coefficients, slack, solver status/residual, saturation, realized cost, and selection/construction/solve/total timing. The model outputs and past-output slack now use all eight physical channels; inputs are NOT appended as extra outputs.

The native stage cost is norm(sqrt(Q_diag)*(y-reference8)/scale_y) + sqrt(r)*norm(u/scale_u), excluding optimization slack. It is not directly comparable with other methods' weighted costs. Timings exclude offline preparation, reference reconstruction, simulation and disk logging; solve-call timing includes CVXPY setup.


In [ ]:
summary_path = RESULT_DIR / 'summary.json'
if summary_path.exists():
    result = json.loads(summary_path.read_text())
    display(pd.DataFrame([{k: v for k, v in result.items() if k not in ['results', 'timing_all_steps', 'timing_discard_first5']}]))
    display(pd.DataFrame(result['results']))
    for key in ['timing_all_steps', 'timing_discard_first5']:
        print(key, '(ms/control step)')
        display(pd.DataFrame(result[key]).T)
else:
    print('No panel results yet:', summary_path)


In [ ]:
PLOT_SEED = 211
PLOT_DIR = RESULT_DIR  # Change to SMOKE_DIR to inspect the short check.
path = PLOT_DIR / f'seed{PLOT_SEED}_control_results.npz'
if path.exists():
    with np.load(path) as a:
        row = json.loads(a['summary'].item())
        if not row['completed_steps']:
            print('No applied steps; failure:', row['failure'])
        else:
            fig, axes = plt.subplots(2, 3, figsize=(14, 7))
            axes[0,0].plot(a['y'][:,6], a['y'][:,7], label='realized')
            axes[0,0].scatter(*a['target_xy'], marker='x', color='red', label='target')
            axes[0,0].axis('equal'); axes[0,0].legend(); axes[0,0].set_title('Fingertip path (m)')
            axes[0,1].plot(a['distance']*1000); axes[0,1].set_title('Target error (mm)')
            axes[0,2].plot(a['prediction_error_xy']*1000); axes[0,2].set_title('One-step prediction error (mm)')
            axes[1,0].plot(a['u']); axes[1,0].plot(a['command_u'], '--', alpha=.5)
            axes[1,0].set_title('Applied (solid) / commanded (dashed) torque')
            axes[1,1].step(np.arange(len(a['cluster'])), a['cluster']); axes[1,1].set_title('Selected cluster')
            for key, label in [('selection_time_s','selection'), ('matrix_preparation_time_s','construction'), ('solve_time_s','solve'), ('control_compute_time_s','total')]:
                axes[1,2].plot(a[key]*1000, label=label)
            axes[1,2].set_title('Online time (ms)'); axes[1,2].legend()
            fig.suptitle(f'MVC-BDPC seed {PLOT_SEED}: {row["completed_steps"]} controlled steps, {row["saturated_steps"]} saturated')
            fig.tight_layout(); plt.show()
else:
    print('No saved rollout:', path)
